In [2]:
# 1 library setup

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import os
import glob
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
from typing import Optional, Tuple, List, Dict
import math


In [3]:
# column def info

ALL_COLUMNS = [
    'HR','O2Sat','Temp','SBP','MAP','DBP','Resp','EtCO2',
    'BaseExcess','HCO3','FiO2','pH','PaCO2','SaO2','AST','BUN',
    'Alkalinephos','Calcium','Chloride','Creatinine','Bilirubin_direct',
    'Glucose','Lactate','Magnesium','Phosphate','Potassium',
    'Bilirubin_total','TroponinI','Hct','Hgb','PTT','WBC','Fibrinogen',
    'Platelets','Age','Gender','Unit1','Unit2','HospAdmTime','ICULOS',
    'SepsisLabel'
]

# Demographics (static or quasi-static)
DEMO_COLS = ['Age', 'Gender', 'Unit1', 'Unit2', 'HospAdmTime']

# Dynamic clinical variables (everything else except label)
LABEL_COL = 'SepsisLabel'

DYNAMIC_COLS = [c for c in ALL_COLUMNS if c not in DEMO_COLS and c != LABEL_COL]

In [4]:
# multi horizon label construction

def make_multihorizon_labels(sepsis_label, horizons=[1,3,6,12]):
    """
    sepsis_label: [T] array, 0/1, with a single transition to 1 at onset
    Returns: Y [T, 4]
    """
    T = len(sepsis_label)
    Y = np.zeros((T, len(horizons)), dtype=np.float32)

    if sepsis_label.max() == 0:
        return Y

    ts = np.where(sepsis_label == 1)[0][0]  # onset time index

    for i, d in enumerate(horizons):
        for t in range(T):
            if t <= ts < t + d:
                Y[t, i] = 1.0

    return Y


In [5]:
# load single patient file

def load_patient_file(path):
    df = pd.read_csv(path, sep='|')

    # Extract dynamic features
    X = df[DYNAMIC_COLS].values.astype(np.float32)

    # Replace missing with NaN (already usually is)
    X[X == -1] = np.nan

    # Demographics: take first non-NaN
    demo = []
    for c in DEMO_COLS:
        col = df[c].values
        idx = np.where(~pd.isna(col))[0]
        if len(idx) == 0:
            demo.append(0.0)
        else:
            demo.append(col[idx[0]])
    demo = np.array(demo, dtype=np.float32)
    demo = np.nan_to_num(demo, nan=0.0, posinf=0.0, neginf=0.0)

    # Sepsis labels
    sepsis = df[LABEL_COL].values.astype(np.int32)

    Y = make_multihorizon_labels(sepsis)

    return X, demo, Y


In [6]:
# load whole dataset

def load_physionet_dataset(root_dir):
    files = sorted(glob.glob(os.path.join(root_dir, "*.psv")))

    patient_X = []
    patient_demo = []
    patient_Y = []

    for f in tqdm(files):
        try:
            X, demo, Y = load_patient_file(f)
            patient_X.append(X)
            patient_demo.append(demo)
            patient_Y.append(Y)
        except Exception as e:
            print("Failed on", f, e)

    return patient_X, patient_demo, patient_Y


In [7]:
# data split on patient level

def split_dataset(patient_X, patient_demo, patient_Y, seed=42):
    N = len(patient_X)
    rng = np.random.RandomState(seed)
    idx = np.arange(N)
    rng.shuffle(idx)

    n_train = int(0.7 * N)
    n_val = int(0.15 * N)

    train_idx = idx[:n_train]
    val_idx = idx[n_train:n_train+n_val]
    test_idx = idx[n_train+n_val:]

    def subset(idxs):
        return (
            [patient_X[i] for i in idxs],
            [patient_demo[i] for i in idxs],
            [patient_Y[i] for i in idxs],
        )

    return subset(train_idx), subset(val_idx), subset(test_idx)


In [8]:
# preprocessing unit

class PhysioNetPreprocessor:
    def __init__(self, pop_mean, pop_std):
        self.pop_mean = pop_mean
        self.pop_std = pop_std

    def transform(self, X, demo=None):
        """
        X: [T, F] with NaNs
        demo: [D] or None
        Returns: [T, 3F + D]
        """
        T, F = X.shape

        mask = (~np.isnan(X)).astype(np.float32)

        delta = np.zeros_like(X, dtype=np.float32)
        last_seen = np.zeros(F)

        for t in range(T):
            for f in range(F):
                if mask[t, f] == 1:
                    delta[t, f] = 0
                    last_seen[f] = t
                else:
                    delta[t, f] = t - last_seen[f]

        delta = np.clip(delta, 0, 72) / 24.0

        # Forward fill
        X_filled = X.copy()
        for f in range(F):
            col = X_filled[:, f]
            idx = np.where(~np.isnan(col))[0]
            if len(idx) == 0:
                X_filled[:, f] = self.pop_mean[f]
            else:
                last = col[idx[0]]
                for t in range(T):
                    if not np.isnan(col[t]):
                        last = col[t]
                    else:
                        col[t] = last

        # Any remaining NaNs? (safety)
        X_filled = np.nan_to_num(X_filled, nan=self.pop_mean)

        X_norm = (X_filled - self.pop_mean) / (self.pop_std + 1e-6)

        parts = [X_norm, mask, delta]
        if demo is not None:
            demo_rep = np.repeat(demo[None, :], T, axis=0)
            parts.append(demo_rep)

        U = np.concatenate(parts, axis=1)
        return torch.tensor(U, dtype=torch.float32)


In [9]:
# dataset wrapper

class PhysioNetDataset(Dataset):
    def __init__(self, patient_X, patient_demo, labels, preprocessor):
        self.patient_X = patient_X
        self.patient_demo = patient_demo
        self.labels = labels  # [T, 4]
        self.prep = preprocessor

    def __len__(self):
        return len(self.patient_X)

    def __getitem__(self, idx):
        X = self.patient_X[idx]
        demo = self.patient_demo[idx]
        Y = self.labels[idx]
        U = self.prep.transform(X, demo)
        return U, torch.tensor(Y, dtype=torch.float32)

In [10]:
# variable length batching

def collate_fn(batch):
    Us, Ys = zip(*batch)
    lengths = [u.shape[0] for u in Us]
    Tm = max(lengths)
    B = len(Us)
    D = Us[0].shape[1]

    U_pad = torch.zeros(B, Tm, D)
    Y_pad = torch.zeros(B, Tm, 4)
    M_pad = torch.zeros(B, Tm)

    for i, (u, y) in enumerate(zip(Us, Ys)):
        T = u.shape[0]
        U_pad[i, :T] = u
        Y_pad[i, :T] = y
        M_pad[i, :T] = 1

    U_pad = torch.nan_to_num(U_pad, nan=0.0, posinf=0.0, neginf=0.0)
    return U_pad, Y_pad, M_pad, torch.tensor(lengths)


In [11]:
# redesigned LNN model

"""
Liquid Neural Network for Sepsis Prediction
Exact GRU Interface: [B, T, D] -> [B, T, 4]
"""

from typing import Optional


class LTCCell(nn.Module):
    """
    Liquid Time-Constant Cell
    Implements continuous-time neural dynamics
    """
    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        ode_unfolds: int = 6,
        epsilon: float = 1e-8
    ):
        super(LTCCell, self).__init__()
        
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.ode_unfolds = ode_unfolds
        self.epsilon = epsilon
        
        # Sensory pathway (input -> hidden)
        self.sensory_mu = nn.Linear(input_size, hidden_size)
        self.sensory_sigma = nn.Linear(input_size, hidden_size)
        self.sensory_w = nn.Linear(input_size, hidden_size)
        
        # Recurrent pathway (hidden -> hidden)
        self.recurrent_mu = nn.Linear(hidden_size, hidden_size)
        self.recurrent_sigma = nn.Linear(hidden_size, hidden_size)
        self.recurrent_w = nn.Linear(hidden_size, hidden_size)
        
        # Time constants
        self.tau = nn.Parameter(torch.ones(hidden_size))
        self.A = nn.Parameter(torch.ones(hidden_size))
        
        self._init_weights()
    
    def _init_weights(self):
        for name, param in self.named_parameters():
            if 'weight' in name:
                nn.init.xavier_uniform_(param)
            elif 'bias' in name:
                nn.init.constant_(param, 0.0)
        nn.init.uniform_(self.tau, 0.5, 2.0)
        nn.init.uniform_(self.A, 0.5, 2.0)
    
    def _sigmoid(self, v_pre, mu, sigma):
        return torch.sigmoid(sigma * (v_pre - mu))
    
    def _ode_step(self, h, x, dt):
        # Sensory input
        sensory_w = self.sensory_w(x)
        sensory_act = self._sigmoid(h, self.sensory_mu(x), self.sensory_sigma(x))
        
        # Recurrent input
        recurrent_w = self.recurrent_w(h)
        recurrent_act = self._sigmoid(h, self.recurrent_mu(h), self.recurrent_sigma(h))
        
        # Combined
        f_h = sensory_w * sensory_act + recurrent_w * recurrent_act
        
        # Time constants (ensure positive)
        tau = F.softplus(self.tau) + self.epsilon
        A = F.softplus(self.A)
        
        # ODE: dh/dt = (1/tau) * (-A * h + f_h)
        dh = (1.0 / tau) * (-A * h + f_h)
        
        # Euler integration
        return h + dt * dh
    
    def forward(self, x, h=None):
        """
        x: [B, D]
        h: [B, H] or None
        returns: [B, H]
        """
        batch_size = x.size(0)
        
        if h is None:
            h = torch.zeros(batch_size, self.hidden_size, device=x.device, dtype=x.dtype)
        
        # ODE integration
        dt = 1.0 / self.ode_unfolds
        for _ in range(self.ode_unfolds):
            h = self._ode_step(h, x, dt)
        
        return h


class LNNSepsisModel(nn.Module):
    """
    Liquid Neural Network for Sepsis Prediction
    
    Interface:
        Input:  U [B, T, D], M [B, T]
        Output: logits [B, T, 4]
    
    Args:
        input_dim: Number of input features (D)
        hidden_dim: Hidden state dimension
        num_layers: Number of LTC layers
        dropout: Dropout probability
        bidirectional: Use bidirectional processing
        ode_unfolds: ODE integration steps (default: 6)
    """
    def __init__(
        self,
        input_dim,
        hidden_dim,
        num_layers=1,
        dropout=0.0,
        bidirectional=False,
        ode_unfolds=6
    ):
        super().__init__()
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1
        
        # Forward LTC cells
        self.forward_cells = nn.ModuleList()
        for layer in range(num_layers):
            cell_input = input_dim if layer == 0 else hidden_dim
            self.forward_cells.append(
                LTCCell(cell_input, hidden_dim, ode_unfolds)
            )
        
        # Backward LTC cells (if bidirectional)
        if bidirectional:
            self.backward_cells = nn.ModuleList()
            for layer in range(num_layers):
                cell_input = input_dim if layer == 0 else hidden_dim
                self.backward_cells.append(
                    LTCCell(cell_input, hidden_dim, ode_unfolds)
                )
        
        # Dropout
        self.dropout = nn.Dropout(dropout) if dropout > 0 else None
        
        # Output head (4 horizons: 1h, 3h, 6h, 12h)
        self.head = nn.Linear(hidden_dim * self.num_directions, 4)
    
    def forward(self, U, M):
        """
        Forward pass
        
        Args:
            U: Input features [B, T, D]
            M: Mask [B, T] (1 for valid, 0 for padding)
        
        Returns:
            logits: [B, T, 4]
        """
        batch_size, seq_len, _ = U.size()
        device = U.device
        dtype = U.dtype
        
        # Initialize hidden states
        h_forward = [
            torch.zeros(batch_size, self.hidden_dim, device=device, dtype=dtype)
            for _ in range(self.num_layers)
        ]
        
        if self.bidirectional:
            h_backward = [
                torch.zeros(batch_size, self.hidden_dim, device=device, dtype=dtype)
                for _ in range(self.num_layers)
            ]
        
        # Forward pass through time
        forward_outputs = []
        for t in range(seq_len):
            x_t = U[:, t]  # [B, D]
            
            
            # Process through layers
            for layer in range(self.num_layers):
                h_t = self.forward_cells[layer](x_t, h_forward[layer])
                h_forward[layer] = h_t
                if self.dropout and layer < self.num_layers - 1:
                    x_t = self.dropout(h_t)
                else:
                    x_t = h_t
            
            logits = self.head(h_t)
            forward_outputs.append(logits)
        
        # Stack: [B, T, H]
        forward_out = torch.stack(forward_outputs, dim=1)
        
        # Backward pass (if bidirectional)
        if self.bidirectional:
            backward_outputs = []
            for t in range(seq_len - 1, -1, -1):
                x_t = U[:, t, :]
                
                for layer in range(self.num_layers):
                    h_t = self.backward_cells[layer](x_t, h_backward[layer])
                    h_backward[layer] = h_t
                    
                    if self.dropout and layer < self.num_layers - 1:
                        x_t = self.dropout(h_t)
                    else:
                        x_t = h_t
                
                backward_outputs.append(h_t)
            
            backward_outputs = backward_outputs[::-1]
            backward_out = torch.stack(backward_outputs, dim=1)
            
            # Concatenate: [B, T, 2*H]
            out = torch.cat([forward_out, backward_out], dim=-1)
        else:
            out = forward_out  # [B, T, H]
        
        # Apply mask (zero out padding)
        mask_expanded = M.unsqueeze(-1)  # [B, T, 1]
        #out = out * mask_expanded
        
        # Project to outputs: [B, T, 4]
        #logits = self.head(out)
        
        return out


'''# Test
if __name__ == "__main__":
    print("Testing LNN Sepsis Model")
    print("=" * 60)
    
    # Configuration
    batch_size = 16
    seq_len = 40
    input_dim = 40
    hidden_dim = 64
    
    # Create model
    model = LNNSepsisModel(
        input_dim=input_dim,
        hidden_dim=hidden_dim,
        num_layers=2,
        dropout=0.2,
        bidirectional=False
    )
    
    print(f"Model created:")
    print(f"  Input dim: {input_dim}")
    print(f"  Hidden dim: {hidden_dim}")
    print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    # Create input
    U = torch.randn(batch_size, seq_len, input_dim)
    M = torch.ones(batch_size, seq_len)
    M[:, 35:] = 0  # Add padding
    
    print(f"\nInput shapes:")
    print(f"  U: {U.shape}")
    print(f"  M: {M.shape}")
    
    # Forward pass
    logits = model(U, M)
    
    print(f"\nOutput shape:")
    print(f"  logits: {logits.shape}")
    print(f"  Expected: torch.Size([{batch_size}, {seq_len}, 4])")
    print(f"  Match: {'✓' if logits.shape == torch.Size([batch_size, seq_len, 4]) else '✗'}")
    
    # Test gradient flow
    target = torch.randn(batch_size, seq_len, 4)
    loss = F.mse_loss(logits, target)
    loss.backward()
    
    grad_exists = all(p.grad is not None for p in model.parameters() if p.requires_grad)
    print(f"\nGradient flow: {'✓' if grad_exists else '✗'}")
    
    print("\n" + "=" * 60)
    print("✓ Model works correctly!")'''

'# Test\nif __name__ == "__main__":\n    print("Testing LNN Sepsis Model")\n    print("=" * 60)\n    \n    # Configuration\n    batch_size = 16\n    seq_len = 40\n    input_dim = 40\n    hidden_dim = 64\n    \n    # Create model\n    model = LNNSepsisModel(\n        input_dim=input_dim,\n        hidden_dim=hidden_dim,\n        num_layers=2,\n        dropout=0.2,\n        bidirectional=False\n    )\n    \n    print(f"Model created:")\n    print(f"  Input dim: {input_dim}")\n    print(f"  Hidden dim: {hidden_dim}")\n    print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")\n    \n    # Create input\n    U = torch.randn(batch_size, seq_len, input_dim)\n    M = torch.ones(batch_size, seq_len)\n    M[:, 35:] = 0  # Add padding\n    \n    print(f"\nInput shapes:")\n    print(f"  U: {U.shape}")\n    print(f"  M: {M.shape}")\n    \n    # Forward pass\n    logits = model(U, M)\n    \n    print(f"\nOutput shape:")\n    print(f"  logits: {logits.shape}")\n    print(f"  Expected: 

In [12]:
# basic LNN like model, uses the correct algorithms but doesn't utilize LTC cells

class MonolithicLNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, tau=4.0, substeps=4):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.tau = tau
        self.substeps = substeps

        self.W = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.V = nn.Linear(input_dim, hidden_dim, bias=True)

        self.readout = nn.Linear(hidden_dim, 4)  # 4 horizons

    def step(self, x, u, dt):
        """
        One Euler step
        """
        dx = -x.double() + torch.tanh(self.W(x) + self.V(u)).double()
        return x + (dt / self.tau) * dx

    def forward(self, U, pad_mask):
        """
        U: [B, T, D]
        pad_mask: [B, T]
        """
        if not torch.isfinite(U).all():
            print("NaNs in input!")
            exit()
        B, T, D = U.shape
        device = U.device

        x = torch.zeros(B, self.hidden_dim, device=device)
        x = x.double()
        outputs = []

        dt = 1.0 / self.substeps

        for t in range(T):
            u_t = U[:, t]
            u_t = u_t.double()
            # sub-stepping solver
            for _ in range(self.substeps):
                x = self.step(x, u_t, dt)

            logits = self.readout(x)
            outputs.append(logits)

        outputs = torch.stack(outputs, dim=1)  # [B, T, 4]
        return outputs


In [13]:
# GRU model

class GRUSepsisModel(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        num_layers=1,
        dropout=0.0,
        bidirectional=False
    ):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1

        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )

        self.head = nn.Linear(hidden_dim * self.num_directions, 4)

    def forward(self, U, M):
        """
        U: [B, T, D]
        M: [B, T]  (1 for valid, 0 for padding)
        """

        # Compute lengths from mask
        lengths = M.sum(dim=1).long().cpu()   # [B]

        # Pack sequence
        packed = nn.utils.rnn.pack_padded_sequence(
            U, lengths, batch_first=True, enforce_sorted=False
        )
        packed = packed.double()

        packed_out, _ = self.gru(packed)

        # Unpack
        out, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out, batch_first=True
        )  # [B, T, H]

        logits = self.head(out)  # [B, T, 4]

        return logits



In [14]:
#LSTM model

class LSTMSepsisModel(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        num_layers=1,
        dropout=0.0,
        bidirectional=False
    ):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )

        self.head = nn.Linear(hidden_dim * self.num_directions, 4)

    def forward(self, U, M):
        """
        U: [B, T, D]
        M: [B, T]  (1 = valid timestep, 0 = padding)
        """

        # Compute valid lengths
        lengths = M.sum(dim=1).long().cpu()

        # Pack padded sequence
        packed = nn.utils.rnn.pack_padded_sequence(
            U, lengths, batch_first=True, enforce_sorted=False
        )
        packed = packed.double()

        packed_out, _ = self.lstm(packed)

        # Unpack back to padded tensor
        out, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out, batch_first=True
        )  # [B, T, H]

        logits = self.head(out)  # [B, T, 4]

        return logits


In [15]:
# reporting for AUROC and AUPRC

@torch.no_grad()
def compute_metrics(model, loader, device):
    model.eval()

    # For each horizon, we accumulate a flat list
    all_probs = {0: [], 1: [], 2: [], 3: []}
    all_targets = {0: [], 1: [], 2: [], 3: []}

    for U, Y, M, L in loader:
        U = U.to(device)
        M = M.to(device)
        U = U.double()
        M = M.double()
        logits = model(U, M)
        probs = torch.sigmoid(logits)  # [B, T, 4]

        B, T, _ = probs.shape

        for k in range(4):
            probs_k = probs[:, :, k]      # [B, T]
            targets_k = Y[:, :, k].to(device)  # [B, T]

            # Use mask to remove padding
            mask = M > 0                  # [B, T]

            probs_k = probs_k[mask]       # [N]
            targets_k = targets_k[mask]   # [N]

            all_probs[k].append(probs_k.cpu())
            all_targets[k].append(targets_k.cpu())

    metrics = {}

    for k, horizon in enumerate([1, 3, 6, 12]):
        probs_k = torch.cat(all_probs[k], dim=0).numpy()
        targets_k = torch.cat(all_targets[k], dim=0).numpy()

        # Remove any NaN pairs
        #valid = np.isfinite(probs_k) & np.isfinite(targets_k)

        #probs_k = probs_k[valid]
        #targets_k = targets_k[valid]

        # Edge case: if only one class present
        if len(np.unique(targets_k)) < 2:
            auroc = np.nan
            auprc = np.nan
        else:
            auroc = roc_auc_score(targets_k, probs_k)
            auprc = average_precision_score(targets_k, probs_k)

        metrics[horizon] = {
            "AUROC": float(auroc),
            "AUPRC": float(auprc)
        }

    # Macro average
    aurocs = [metrics[h]["AUROC"] for h in [1,3,6,12] if not np.isnan(metrics[h]["AUROC"])]
    auprcs = [metrics[h]["AUPRC"] for h in [1,3,6,12] if not np.isnan(metrics[h]["AUPRC"])]

    metrics["macro"] = {
        "AUROC": float(np.mean(aurocs)) if len(aurocs) else np.nan,
        "AUPRC": float(np.mean(auprcs)) if len(auprcs) else np.nan,
    }

    return metrics

def print_metrics(metrics, prefix=""):
    line = prefix
    for h in [1,3,6,12]:
        m = metrics[h]
        line += f" | {h}h AUROC {m['AUROC']:.3f} AUPRC {m['AUPRC']:.3f}"
    line += f" | Macro AUPRC {metrics['macro']['AUPRC']:.3f}"
    print(line)

In [16]:
# multi horizon loss function

def multi_horizon_loss(logits, targets, mask, pos_weights):
    """
    logits: [B, T, 4]
    targets: [B, T, 4]
    mask: [B, T]
    pos_weights: [4]
    """
    loss = 0
    for k in range(4):
        bce = F.binary_cross_entropy_with_logits(
            logits[:, :, k],
            targets[:, :, k],
            reduction="none",
            pos_weight=pos_weights[k]
        )
        bce = bce * mask
        loss += bce.sum() / mask.sum()

    return loss


In [17]:
# training loop

def train_epoch(model, loader, optimizer, device, pos_weights):
    model.double()
    model.train()
    total = 0

    for U, Y, M, L in tqdm(loader):
        U = U.to(device)
        Y = Y.to(device)
        M = M.to(device)

        U = U.double()
        Y = Y.double()
        M = M.double()

        optimizer.zero_grad()
        logits = model(U, M)
        loss = multi_horizon_loss(logits, Y, M, pos_weights)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        total += loss.item()

    return total / len(loader)


In [18]:
# validation

@torch.no_grad()
def eval_epoch(model, loader, device):
    model.double()
    model.eval()
    all_logits = []
    all_targets = []
    all_masks = []

    for U, Y, M, L in loader:
        U = U.to(device)
        U = U.double()
        Y = Y.double()
        M = M.double()
        logits = model(U, M)

        all_logits.append(torch.sigmoid(logits).cpu())
        all_targets.append(Y)
        all_masks.append(M)

    return torch.cat(all_logits), torch.cat(all_targets), torch.cat(all_masks)


In [19]:
# check to see if dataset contains nans before our run

def check_dataset_for_nans(train_X, train_demo, train_Y, name="train"):
    print(f"Checking {name} dataset...")

    bad = False

    # Check X
    for i, X in enumerate(train_X):
        if not np.isfinite(X).all():
            idx = np.where(~np.isfinite(X))
            print(f"[{name}] NaN/Inf in train_X[{i}] at positions {list(zip(idx[0][:5], idx[1][:5]))} ...")
            bad = True
            break

    # Check demo
    for i, d in enumerate(train_demo):
        if not np.isfinite(d).all():
            idx = np.where(~np.isfinite(d))
            print(f"[{name}] NaN/Inf in train_demo[{i}] at positions {idx[0][:5]} ...")
            bad = True
            break

    # Check Y
    for i, Y in enumerate(train_Y):
        if not np.isfinite(Y).all():
            idx = np.where(~np.isfinite(Y))
            print(f"[{name}] NaN/Inf in train_Y[{i}] at positions {list(zip(idx[0][:5], idx[1][:5]))} ...")
            bad = True
            break

    if not bad:
        print(f"[{name}] No NaNs or Infs found in dataset.")
    else:
        print(f"[{name}] Dataset contains invalid values.")

# as it currently stands, the dataset does still contain nans, 
# but I have fixes later on that make sure internal model function still goes smoothly


In [20]:
# memory tracking

def reset_cuda_memory_stats():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()

In [21]:
# code run for data load


DATA_DIR = "data/physionet/training_setA/training"

patient_X, patient_demo, patient_Y = load_physionet_dataset(DATA_DIR)

(train_X, train_demo, train_Y), (val_X, val_demo, val_Y), (test_X, test_demo, test_Y) = \
    split_dataset(patient_X, patient_demo, patient_Y)

check_dataset_for_nans(train_X, train_demo, train_Y, name="train")
check_dataset_for_nans(val_X, val_demo, val_Y, name="val")
check_dataset_for_nans(test_X, test_demo, test_Y, name="test")

print("Train patients:", len(train_X))
print("Val patients:", len(val_X))
print("Test patients:", len(test_X))

#num_features = train_X[0].shape[1]
#demo_dim = train_demo[0].shape[0]

pop_mean = np.nanmean(np.concatenate(train_X, axis=0), axis=0)
pop_std  = np.nanstd(np.concatenate(train_X, axis=0), axis=0)

pop_mean = np.nan_to_num(pop_mean, nan=0.0)
pop_std = np.nan_to_num(pop_std, nan=1.0)
pop_std[pop_std < 1e-6] = 1.0

prep = PhysioNetPreprocessor(pop_mean, pop_std)

ds_train = PhysioNetDataset(train_X, train_demo, train_Y, prep)
ds_val   = PhysioNetDataset(val_X, val_demo, val_Y, prep)

dl_train = DataLoader(ds_train, batch_size=32, shuffle=True, collate_fn=collate_fn)
dl_val   = DataLoader(ds_val, batch_size=32, shuffle=False, collate_fn=collate_fn)

100%|██████████| 20336/20336 [04:30<00:00, 75.17it/s]


Checking train dataset...
[train] NaN/Inf in train_X[0] at positions [(0, 7), (0, 10), (0, 12), (0, 13), (0, 14)] ...
[train] Dataset contains invalid values.
Checking val dataset...
[val] NaN/Inf in train_X[0] at positions [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4)] ...
[val] Dataset contains invalid values.
Checking test dataset...
[test] NaN/Inf in train_X[0] at positions [(0, 7), (0, 13), (0, 17), (0, 19), (0, 20)] ...
[test] Dataset contains invalid values.
Train patients: 14235
Val patients: 3050
Test patients: 3051


C:\Users\david\AppData\Local\Temp\ipykernel_22244\1675536066.py:22: RuntimeWarning: Mean of empty slice
  pop_mean = np.nanmean(np.concatenate(train_X, axis=0), axis=0)
C:\Users\david\AppData\Roaming\Python\Python310\site-packages\numpy\lib\nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


In [ ]:
# LNN run
device = "cuda"

'''model = MonolithicLNN(
    input_dim=ds_train[0][0].shape[1],
    hidden_dim=256,
    tau=4.0,
    substeps=4
).to(device)'''
model = LNNSepsisModel(
    input_dim=ds_train[0][0].shape[1],
    hidden_dim=256,
    ode_unfolds=4,
    num_layers=2
).to(device)
model.double()

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

pos_weights = torch.tensor([10.0, 5.0, 3.0, 2.0]).to(device)

best = -1
patience = 10
bad_epochs = 0

if torch.cuda.is_available():
    reset_cuda_memory_stats()

for epoch in range(20):
    train_loss = train_epoch(model, dl_train, optimizer, device, pos_weights)    
    print("Epoch", epoch, "train loss", train_loss)

    val_metrics = compute_metrics(model, dl_val, device)

    if val_metrics["macro"]["AUPRC"] > best:
        best = val_metrics["macro"]["AUPRC"]
        bad_epochs = 0
        torch.save(model.state_dict(), "best_physio/best_lnn.pt")
    else:
        bad_epochs += 1
        if bad_epochs >= patience:
            print("Early stopping!")
            break

    print_metrics(val_metrics, prefix="Val")

if torch.cuda.is_available():
    max_alloc = torch.cuda.max_memory_allocated() / 1024**2
    max_reserved = torch.cuda.max_memory_reserved() / 1024**2

    print(f"\nGPU Memory Usage:")
    print(f"  Max allocated: {max_alloc:.2f} MB")
    print(f"  Max reserved : {max_reserved:.2f} MB")

100%|██████████| 445/445 [17:53<00:00,  2.41s/it]


Epoch 0 train loss 0.6987206091879321
Val | 1h AUROC 0.600 AUPRC 0.004 | 3h AUROC 0.548 AUPRC 0.008 | 6h AUROC 0.556 AUPRC 0.016 | 12h AUROC 0.574 AUPRC 0.031 | Macro AUPRC 0.015


100%|██████████| 445/445 [18:38<00:00,  2.51s/it]


Epoch 1 train loss 0.6676359984353043
Val | 1h AUROC 0.647 AUPRC 0.008 | 3h AUROC 0.696 AUPRC 0.016 | 6h AUROC 0.684 AUPRC 0.031 | 12h AUROC 0.646 AUPRC 0.071 | Macro AUPRC 0.032


100%|██████████| 445/445 [17:04<00:00,  2.30s/it]


...

100%|██████████| 445/445 [16:57<00:00,  2.29s/it]


Epoch 11 train loss 0.5386478892690157
Val | 1h AUROC 0.737 AUPRC 0.012 | 3h AUROC 0.710 AUPRC 0.020 | 6h AUROC 0.724 AUPRC 0.041 | 12h AUROC 0.729 AUPRC 0.078 | Macro AUPRC 0.038


100%|██████████| 445/445 [17:19<00:00,  2.34s/it]


Epoch 12 train loss 0.5367179818212331
Val | 1h AUROC 0.754 AUPRC 0.010 | 3h AUROC 0.750 AUPRC 0.022 | 6h AUROC 0.748 AUPRC 0.039 | 12h AUROC 0.765 AUPRC 0.078 | Macro AUPRC 0.037


 21%|██        | 93/445 [03:46<13:10,  2.25s/it]

In [57]:
# GRU run for comparison

gru_model = GRUSepsisModel(
    input_dim=ds_train[0][0].shape[1],
    hidden_dim=256, 
    num_layers=1,
    dropout=0.0,
    bidirectional=False
).to(device)
gru_model.double()

optimizer = torch.optim.AdamW(gru_model.parameters(), lr=3e-4, weight_decay=1e-4)

best = -1
patience = 10
bad_epochs = 0

if torch.cuda.is_available():
    reset_cuda_memory_stats()

for epoch in range(20):
    train_loss = train_epoch(gru_model, dl_train, optimizer, device, pos_weights)    
    print("Epoch", epoch, "train loss", train_loss)

    val_metrics = compute_metrics(gru_model, dl_val, device)
    if val_metrics["macro"]["AUPRC"] > best:
        best = val_metrics["macro"]["AUPRC"]
        bad_epochs = 0
        torch.save(gru_model.state_dict(), "best_physio/best_gru.pt")
    else:
        bad_epochs += 1
        if bad_epochs >= patience:
            print("Early stopping!")
            break
    print_metrics(val_metrics, prefix="Val")

if torch.cuda.is_available():
    max_alloc = torch.cuda.max_memory_allocated() / 1024**2
    max_reserved = torch.cuda.max_memory_reserved() / 1024**2

    print(f"\nGPU Memory Usage:")
    print(f"  Max allocated: {max_alloc:.2f} MB")
    print(f"  Max reserved : {max_reserved:.2f} MB")

  7%|▋         | 33/445 [03:15<40:35,  5.91s/it]   


KeyboardInterrupt: 

In [ ]:
# LSTM run for comparison

lstm_model = LSTMSepsisModel(
    input_dim=ds_train[0][0].shape[1],
    hidden_dim=256,
    num_layers=1,
    dropout=0.0,
    bidirectional=False
).to(device)
lstm_model.double()

optimizer = torch.optim.AdamW(lstm_model.parameters(), lr=3e-4, weight_decay=1e-4)

best = -1
patience = 10
bad_epochs = 0

if torch.cuda.is_available():
    reset_cuda_memory_stats()

for epoch in range(20):
    train_loss = train_epoch(lstm_model, dl_train, optimizer, device, pos_weights)    
    print("Epoch", epoch, "train loss", train_loss)

    val_metrics = compute_metrics(lstm_model, dl_val, device)
    if val_metrics["macro"]["AUPRC"] > best:
        best = val_metrics["macro"]["AUPRC"]
        bad_epochs = 0
        torch.save(lstm_model.state_dict(), "best_physio/best_lstm.pt")
    else:
        bad_epochs += 1
        if bad_epochs >= patience:
            print("Early stopping!")
            break
    print_metrics(val_metrics, prefix="Val")

if torch.cuda.is_available():
    max_alloc = torch.cuda.max_memory_allocated() / 1024**2
    max_reserved = torch.cuda.max_memory_reserved() / 1024**2

    print(f"\nGPU Memory Usage:")
    print(f"  Max allocated: {max_alloc:.2f} MB")
    print(f"  Max reserved : {max_reserved:.2f} MB")

100%|██████████| 445/445 [01:28<00:00,  5.04it/s]


Epoch 0 train loss 0.6412679123329675
Val | 1h AUROC 0.718 AUPRC 0.007 | 3h AUROC 0.709 AUPRC 0.018 | 6h AUROC 0.707 AUPRC 0.037 | 12h AUROC 0.712 AUPRC 0.074 | Macro AUPRC 0.034


100%|██████████| 445/445 [01:27<00:00,  5.07it/s]


Epoch 1 train loss 0.5135292129496308
Val | 1h AUROC 0.760 AUPRC 0.008 | 3h AUROC 0.745 AUPRC 0.022 | 6h AUROC 0.739 AUPRC 0.042 | 12h AUROC 0.743 AUPRC 0.082 | Macro AUPRC 0.039


100%|██████████| 445/445 [01:27<00:00,  5.07it/s]


...

Epoch 17 train loss 0.4548455296596211
Val | 1h AUROC 0.791 AUPRC 0.012 | 3h AUROC 0.778 AUPRC 0.025 | 6h AUROC 0.781 AUPRC 0.047 | 12h AUROC 0.785 AUPRC 0.090 | Macro AUPRC 0.044


100%|██████████| 445/445 [01:27<00:00,  5.06it/s]


Epoch 18 train loss 0.4491057532409078
Val | 1h AUROC 0.794 AUPRC 0.014 | 3h AUROC 0.783 AUPRC 0.027 | 6h AUROC 0.784 AUPRC 0.050 | 12h AUROC 0.788 AUPRC 0.095 | Macro AUPRC 0.046


100%|██████████| 445/445 [01:27<00:00,  5.06it/s]


Epoch 19 train loss 0.4460507188499486
Val | 1h AUROC 0.793 AUPRC 0.013 | 3h AUROC 0.783 AUPRC 0.026 | 6h AUROC 0.784 AUPRC 0.048 | 12h AUROC 0.792 AUPRC 0.092 | Macro AUPRC 0.045

GPU Memory Usage:
  Max allocated: 113.11 MB
  Max reserved : 218.00 MB
